## Data vizualization
In this notebook, I will try toimplement meaningful visualization to better understand the structure of the data from patient 3455.
Inspired from subjects_loop.ipynb

In [36]:
# Imports
import os
import numpy as np
import mne
from mne_bids import (
    find_matching_paths,
    read_raw_bids,
)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mne.stats import permutation_cluster_test

from mne import Epochs
from mne.time_frequency import EpochsTFRArray

from mne.datasets import fetch_fsaverage

from pathlib import Path
import mne_bids
from mne_bids import BIDSPath

import matplotlib 
matplotlib.use('Agg') # or nbAgg
mne.viz.set_3d_backend('notebook')

'notebook'

In [37]:
#sample_path = "/media/RCPNAS/sEEG_MARS_Alison/"
subjects_dir = Path("/media/RCPNAS/sEEG_MARS_Alison")

In [38]:
# Your BIDS root
bids_root = subjects_dir # / "BIDS"

# FreeSurfer subjects directory - this is the key path!
fs_subjects_dir = Path("/media/RCPNAS/sEEG_MARS_Alison/sourcedata/reconstructions") #bids_root / "sourcedata" / "reconstructions" / "PAT_3455" / "surf"

# Verify the PAT3455 subject exists
pat3455_path = fs_subjects_dir / "PAT_3455" #Path(subjects_dir) / "elec_recon" #bids_root / "sourcedata" / "reconstructions"
print(f"PAT3455 path exists: {pat3455_path.exists()}")
if pat3455_path.exists():
    print("Found FreeSurfer subject PAT3455 with directories:")
    print([d.name for d in pat3455_path.iterdir() if d.is_dir()])

# TODO: Now create mapping for all subjects
# Based on your folder structure, we need to map each BIDS subject to the correct patient
subject_to_fs = {
    "02": "PAT_3455",  #
}

# Process sub-01
subject = "02"
session = "retrieval"
task = "mars"
fs_subject = subject_to_fs[subject]

print(f"\nProcessing {subject} → FreeSurfer subject: {fs_subject}")
print(f"FreeSurfer directory: {fs_subjects_dir / fs_subject}")

# Create BIDS path
bids_path = BIDSPath(
    subject=subject,
    session=session,
    task=task,
    datatype="ieeg",
    root=bids_root,
    suffix="ieeg",
    extension=".vhdr"
)

# Read data
print(f"\nLoading data from: {bids_path.fpath}")
raw = read_raw_bids(bids_path, verbose=False)

# Load bad annotations
bad_annotation_path = bids_root / f"sub-{subject}" / f"ses-{session}" / "ieeg" / f"sub-{subject}_ses-{session}_task-{task}_annot.csv"
if bad_annotation_path.exists():
    print("Bad annotation file found, adding to raw annotations")
    bad_annotations = mne.read_annotations(bad_annotation_path)
    #bad_annotations.set_orig_time(None)
    #bad_annotations.orig_time = raw.annotations.orig_time
    # Recreate annotations with matching orig_time
    bad_annotations = mne.Annotations(
        onset=bad_annotations.onset,
        duration=bad_annotations.duration,
        description=bad_annotations.description,
        orig_time=raw.annotations.orig_time
    )
    
    raw.set_annotations(raw.annotations + bad_annotations)
    print(f"Added {len(bad_annotations)} bad annotations")

# Now visualize electrodes on brain
print(f"\nCreating brain visualization for {fs_subject}")

# Create brain
brain = mne.viz.Brain(
    fs_subject,
    subjects_dir=str(fs_subjects_dir),  # This points to the freesurfer folder containing HL1996
    cortex="low_contrast",
    alpha=0.25,
    background="white",
    figure=1,
)

# Add electrodes if montage exists
if raw.get_montage() is not None:
    try:
        trans = mne.transforms.Transform('mri', 'head')  # Use identity transform for now
        brain.add_sensors(raw.info, trans=trans)
        print("Added electrodes to brain")
    except Exception as e:
        print(f"Could not add electrodes: {e}")
montage = raw.get_montage()
print(f"Montage exists: {montage is not None}")

if montage is None:
    print("\nNo montage found in raw data.")
    print("Looking for electrodes.tsv file...")
    
    # Look for electrodes.tsv
    electrodes_path = bids_root / f"sub-{subject}" / f"ses-{session}" / "ieeg" / f"sub-{subject}_ses-{session}_task-{task}_electrodes.tsv"
    print(f"Electrodes file exists: {electrodes_path.exists()}")
    
    if electrodes_path.exists():
        # Read electrodes.tsv
        electrodes_df = pd.read_csv(electrodes_path, sep='\t')
        print(f"\nElectrodes file contents:")
        print(f"Columns: {electrodes_df.columns.tolist()}")
        print(f"Number of electrodes: {len(electrodes_df)}")
        print("\nFirst few electrodes:")
        print(electrodes_df.head())
        
        # Check coordinate columns
        coord_cols = ['x', 'y', 'z']
        if all(col in electrodes_df.columns for col in coord_cols):
            print("\n✓ Found coordinate columns (x, y, z)")
            
            # Create montage
            montage = mne.channels.make_dig_montage(
                ch_pos=dict(zip(electrodes_df['name'], 
                               electrodes_df[coord_cols].values)),
                coord_frame='mri'  # or 'head' - check which one
            )
            raw.set_montage(montage)
            print(f"✓ Created and set montage with {len(electrodes_df)} electrodes")
        else:
            print(f"✗ Expected coordinate columns not found. Available columns: {electrodes_df.columns.tolist()}")
    else:
        print("✗ No electrodes.tsv file found in BIDS directory")
        
        # Look for alternative coordinate files
        coord_files = list(Path(bids_root / f"sub-{subject}").rglob("*coord*"))
        print(f"Alternative coordinate files found: {coord_files}")
#brain.add_head(alpha=0.25, color="tan")

# Save images
output_dir = Path.cwd() / "figures"
output_dir.mkdir(exist_ok=True)

# Save different views
views = [
    ("lateral_right", dict(azimuth=90, elevation=90, distance=400)),
    ("lateral_left", dict(azimuth=-90, elevation=90, distance=400)),
    ("top", dict(azimuth=0, elevation=90, distance=400)),
    ("front", dict(azimuth=0, elevation=0, distance=400)),
    ("back", dict(azimuth=180, elevation=0, distance=400)),
]

for view_name, view_kwargs in views:
    brain.show_view(**view_kwargs)
    output_file = output_dir / f"{subject}_electrodes_{view_name}.png"
    brain.save_image(str(output_file))
    print(f"Saved: {output_file}")

brain.close()
print(f"\nAll images saved to {output_dir}")

PAT3455 path exists: True
Found FreeSurfer subject PAT3455 with directories:
['mri', 'tmp', 'stats', 'surf_old', 'touch', 'trash', 'surf', 'voxeloc', 'elec_recon_pierre_voxeloc', 'label', 'mri_old', 'scripts', 'elec_recon', 'label_old', 'elec_recon_old']

Processing 02 → FreeSurfer subject: PAT_3455
FreeSurfer directory: /media/RCPNAS/sEEG_MARS_Alison/sourcedata/reconstructions/PAT_3455

Loading data from: /media/RCPNAS/sEEG_MARS_Alison/sub-02/ses-retrieval/ieeg/sub-02_ses-retrieval_task-mars_ieeg.vhdr
Bad annotation file found, adding to raw annotations
Added 1 bad annotations

Creating brain visualization for PAT_3455
Could not estimate rigid Talairach alignment, using identity matrix


Channel types::	seeg: 115
Added electrodes to brain
Montage exists: True
Saved: /home/aboschun/MIPlab-Project/figures/02_electrodes_lateral_right.png
Saved: /home/aboschun/MIPlab-Project/figures/02_electrodes_lateral_left.png
Saved: /home/aboschun/MIPlab-Project/figures/02_electrodes_top.png
Saved: /home/aboschun/MIPlab-Project/figures/02_electrodes_front.png
Saved: /home/aboschun/MIPlab-Project/figures/02_electrodes_back.png

All images saved to /home/aboschun/MIPlab-Project/figures


In [39]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne
from mne_bids import BIDSPath, read_raw_bids

# Your paths
bids_root = Path("/media/RCPNAS/sEEG_MARS_Alison")

# For PAT_3455, figure out which BIDS subject this is
# Based on your earlier mapping, PAT_3455 might be sub-03 or sub-04
# Let's check participants.tsv if it exists
participants_file = bids_root / "participants.tsv"
if participants_file.exists():
    participants = pd.read_csv(participants_file, sep='\t')
    print("Participants mapping:")
    print(participants)

# For now, let's assume PAT_3455 is sub-03 (adjust based on your mapping)
subject = "02"  # Change this based on actual mapping
session = "retrieval"
task = "mars"

# Load raw data to get channel names in TRC order
bids_path = BIDSPath(
    subject=subject,
    session=session,
    task=task,
    datatype="ieeg",
    root=bids_root,
    suffix="ieeg",
    extension=".vhdr"
)

raw = read_raw_bids(bids_path, verbose=False)
trc_channel_names = raw.ch_names
#trc_channel_names = ['HAD1', 'HAD2', 'HAG1', 'HAG2', 'HPD1', 'HPD2', 'HPG1', 'HPG2']  
print(f"TRC channel names (first 10): {trc_channel_names[:10]}")
print(f"Total channels: {len(trc_channel_names)}")

# Path to electrode reconstruction files
elec_recon_path = bids_root / "sourcedata" / "reconstructions" / "PAT_3455" / "elec_recon"

coord_type = "LEPTO"  #     use LEPTO !!!!!
coord_file = elec_recon_path / f"PAT_3455.{coord_type}"

print(f"\nLoading coordinates from: {coord_file}")
print(f"File exists: {coord_file.exists()}")

if coord_file.exists():
    # Read the coordinate file
    # These are typically text files with coordinates in mm
    coords = []
    with open(coord_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):  # Skip empty lines and comments
                try:
                    # Parse x y z coordinates
                    parts = line.split()
                    if len(parts) >= 3:
                        x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
                        coords.append([x, y, z])
                except:
                    continue
    
    coords = np.array(coords)
    print(f"Loaded {len(coords)} coordinates")
    
    # Also load electrode names if available
    names_file = elec_recon_path / "PAT_3455.electrodeNames"
    if names_file.exists():
        with open(names_file, 'r') as f:
            electrode_names = [line.strip() for line in f if line.strip() and not line.startswith('#')]
        print(f"Loaded {len(electrode_names)} electrode names")
    else:
        electrode_names = [f"ELEC{i+1:03d}" for i in range(len(coords))]
    
    # Check if number matches TRC channels
    print(f"\nCoordinate count: {len(coords)}")
    print(f"TRC channel count: {len(trc_channel_names)}")
    print(f"Electrode names count: {len(electrode_names)}")
    
    if len(coords) == len(trc_channel_names):
        print("✓ Coordinate count matches TRC channels - using as-is")
        montage_coords = coords
        montage_names = trc_channel_names
    else:
        print("⚠️  Count mismatch - need to map coordinates to TRC channels")
        
        # Create mapping dictionary
        coord_dict = dict(zip(electrode_names, coords))
        
        # Map to TRC order
        montage_coords = []
        montage_names = []
        
        for ch_name in trc_channel_names:
            # Try exact match
            #if ch_name not in ["HAD1", "HAD2", "HAG1", "HAG2", "HPD1", "HPD2", "HPG1", "HPG2"] :
            #    continue
            if ch_name in coord_dict:
                montage_coords.append(coord_dict[ch_name])
                montage_names.append(ch_name)
            else:
                # Try without spaces/special chars
                ch_clean = ch_name.replace(' ', '').replace('-', '').replace("'", "")
                found = False
                for coord_name in coord_dict.keys():
                    coord_clean = coord_name.replace(' ', '').replace('-', '').replace("'", "")
                    if ch_clean in coord_clean or coord_clean in ch_clean:
                        montage_coords.append(coord_dict[coord_name])
                        montage_names.append(ch_name)
                        print(f"  Matched {ch_name} to {coord_name}")
                        found = True
                        break
                
                if not found:
                    print(f"  Warning: No match for {ch_name}")
                    montage_coords.append([np.nan, np.nan, np.nan])
                    montage_names.append(ch_name)
        
        montage_coords = np.array(montage_coords)
    montage_coords = montage_coords / 1000  # Convert mm to m for MNE
    
    # Create montage
    montage = mne.channels.make_dig_montage(
        ch_pos=dict(zip(montage_names, montage_coords)),
        coord_frame='mri'  # These are in MRI coordinates
    )
    
    # Set montage to raw
    raw.set_montage(montage)
    print(f"\n✓ Created montage with {len(montage_names)} electrodes")
    
    # Verify first few electrodes
    print("\nFirst 5 electrode positions:")
    for i, ch_name in enumerate(montage_names[:5]):
        pos = montage.get_positions()['ch_pos'][ch_name]
        print(f"  {ch_name}: ({pos[0]}, {pos[1]:.1f}, {pos[2]:.1f})")
    
    # Save montage for future use
    montage_path = Path.cwd() / f"sub-{subject}_montage.fif"
    montage.save(montage_path, overwrite=True)
    print(f"Saved montage to {montage_path}")
    
    # Now you can get volume labels if FreeSurfer subject exists
    fs_subject = "PAT_3455"
    subjects_dir = bids_root / "sourcedata" / "reconstructions"
    
    aparcaseg_path = "/media/RCPNAS/sEEG_MARS_Alison/sourcedata/reconstructions/PAT_3455/mri/aparc+aseg.mgz"

    if Path(aparcaseg_path).exists():
        try:
            labels, colors = mne.get_montage_volume_labels(
                montage,
                fs_subject,
                subjects_dir=str(subjects_dir),
                aseg="aparc+aseg"
            )
            
            # labels and colors are dicts keyed by channel name
            # Build DataFrame from the dicts directly
            results_df = pd.DataFrame({
                'channel': list(labels.keys()),
                'label': [labels[ch] for ch in labels.keys()],
                'color': [colors[ch] for ch in labels.keys()]
            })
            
            output_path = Path.cwd() / f"sub-{subject}_electrode_labels.csv"
            results_df.to_csv(output_path, index=False)
            print(f"Saved electrode labels to {output_path}")
            print(results_df.head())
            
        except Exception as e:
            print(f"Could not get volume labels: {e}")

Participants mapping:
  participant_id  age  sex  hand  weight  height
0         sub-01  NaN  NaN   NaN     NaN     NaN
1         sub-02  NaN  NaN   NaN     NaN     NaN
2         sub-03  NaN  NaN   NaN     NaN     NaN
3         sub-04  NaN  NaN   NaN     NaN     NaN
4         sub-06  NaN  NaN   NaN     NaN     NaN
TRC channel names (first 10): ['AD1', 'AD2', 'AD3', 'AD4', 'AD5', 'AD6', 'AD7', 'AD8', 'HAD1', 'HAD2']
Total channels: 120

Loading coordinates from: /media/RCPNAS/sEEG_MARS_Alison/sourcedata/reconstructions/PAT_3455/elec_recon/PAT_3455.LEPTO
File exists: True
Loaded 116 coordinates
Loaded 118 electrode names

Coordinate count: 116
TRC channel count: 120
Electrode names count: 118
⚠️  Count mismatch - need to map coordinates to TRC channels
  Matched AD1 to AD1 D R
  Matched AD2 to AD2 D R
  Matched AD3 to AD3 D R
  Matched AD4 to AD4 D R
  Matched AD5 to AD5 D R
  Matched AD6 to AD6 D R
  Matched AD7 to AD7 D R
  Matched AD8 to AD8 D R
  Matched HAD1 to HAD1 D R
  Matched HA

/tmp/ipykernel_2602962/2422133623.py:133: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw.set_montage(montage)
/tmp/ipykernel_2602962/2422133623.py:133: RuntimeWarning: Not setting positions of 4 ecg/stim channels found in montage:
['MKR1+', 'photodiode', 'MKR2+', 'ECG']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw.set_montage(montage)
/tmp/ipykernel_2602962/2422133623.py:144: RuntimeWarning: This filename (/home/aboschun/MIPlab-Project/sub-02_montage.fif) does not conform to MNE naming conventions. All montage files should end with -dig.fif or -dig.fif.gz
  montage.save(montage_path, overwrite=True)


Could not get volume labels: 'AD1'


In [40]:
montage.get_positions()

{'ch_pos': OrderedDict([('AD1',
               array([ 0.02545176, -0.00344268, -0.00522475])),
              ('AD2', array([ 0.02867764, -0.00266402, -0.00633712])),
              ('AD3', array([ 0.03190353, -0.00188536, -0.00744949])),
              ('AD4', array([ 0.03512941, -0.00110669, -0.00856186])),
              ('AD5', array([ 0.03835529, -0.00032803, -0.00967424])),
              ('AD6', array([ 0.04158117,  0.00045063, -0.01078661])),
              ('AD7', array([ 0.021, -0.019, -0.003])),
              ('AD8', array([ 0.02427734, -0.01780824, -0.00329794])),
              ('HAD1', array([ 0.02755468, -0.01661648, -0.00359588])),
              ('HAD2', array([ 0.03083201, -0.01542472, -0.00389382])),
              ('HAD3', array([ 0.03410935, -0.01423296, -0.00419176])),
              ('HAD4', array([ 0.03738669, -0.0130412 , -0.0044897 ])),
              ('HAD5', array([ 0.04066403, -0.01184945, -0.00478764])),
              ('HAD6', array([ 0.04394136, -0.01065769, -0.005

In [56]:
aseg = "aparc+aseg"  # parcellation/anatomical segmentation atlas
path_atlas = "/media/RCPNAS/sEEG_MARS_Alison/sourcedata/reconstructions"
montage = raw.get_montage()

# Get volume labels for each electrode based on the montage and FreeSurfer subject
labels, colors = mne.get_montage_volume_labels(
    montage, "PAT_3455", subjects_dir=path_atlas, aseg=aseg
)
print(f"Labels: {labels}")

# separate by electrodes which have names like LAMY 1
electrodes = set(
    [
        "".join([lttr for lttr in ch_name if not lttr.isdigit() and lttr != " "])
        for ch_name in montage.ch_names
    ]
)
print(f"Electrodes in the dataset: {electrodes}")

# Define regions of interest (ROIs) to keep
regions_of_interest = {
    "ctx-rh-parahippocampal",
    "ctx-rh-fusiform",
    "Right-Hippocampus",
    "ctx-lh-parahippocampal",
    "ctx-lh-fusiform",
    "Left-Hippocampus",
    # add more as needed
}

# Get labels that correspond to the ROIs --> electrodes that are in the regions of interest
filtered_labels = {
    ch: region_list
    for ch, region_list in labels.items()
    if any(r in regions_of_interest for r in region_list)
}

print(f"Kept {len(filtered_labels)}/{len(labels)} electrodes")
for ch, regions in filtered_labels.items():
    print(f"  {ch}: {regions}")

# New set of electrodes that have labels in the ROIs
filtered_electrodes = set([ch.split(" ")[0] for ch in filtered_labels.keys()])

for elec in filtered_electrodes:
    picks = [ch_name for ch_name in  filtered_labels.keys() if elec in ch_name] #[elec] 
    print(picks)
    if not picks:
        continue  # skip electrodes with no ROI channels
    fig, ax = mne.viz.plot_channel_labels_circle(filtered_labels, colors, picks=picks)
    fig.text(0.3, 0.9, "Anatomical Labels", color="white")
    fig.savefig(f"figures/{subject}_{elec}_labels.png", dpi=300)

Labels: OrderedDict({'AD1': ['Right-Cerebral-White-Matter', 'Right-Amygdala'], 'AD2': ['Right-Cerebral-White-Matter'], 'AD3': ['Right-Cerebral-White-Matter'], 'AD4': ['Right-Cerebral-White-Matter', 'ctx-rh-middletemporal'], 'AD5': ['Right-Cerebral-White-Matter', 'ctx-rh-middletemporal'], 'AD6': ['Right-Cerebral-White-Matter', 'ctx-rh-middletemporal'], 'AD7': ['Right-Cerebral-White-Matter', 'Right-Hippocampus'], 'AD8': ['Right-Cerebral-White-Matter', 'Right-Hippocampus'], 'HAD1': ['Right-Cerebral-White-Matter', 'Right-Inf-Lat-Vent', 'Right-Hippocampus'], 'HAD2': ['Right-Cerebral-White-Matter', 'Right-Inf-Lat-Vent', 'Right-Hippocampus', 'ctx-rh-fusiform'], 'HAD3': ['Unknown', 'Right-Cerebral-White-Matter', 'Right-Inf-Lat-Vent'], 'HAD4': ['Right-Cerebral-White-Matter'], 'HAD5': ['Right-Cerebral-White-Matter'], 'HAD6': ['Right-Cerebral-White-Matter', 'ctx-rh-inferiortemporal'], 'HAD7': ['Right-Cerebral-White-Matter', 'ctx-rh-middletemporal'], 'HAD8': ['Right-Cerebral-White-Matter', 'ctx-rh

In [54]:
epochs = mne.Epochs(raw, detrend=1, baseline=None)
epochs

Used Annotations descriptions: [np.str_('1'), np.str_('2'), np.str_('3'), np.str_('4'), np.str_('5'), np.str_('6'), np.str_('7')]
Ignoring annotation durations and creating fixed-duration epochs around annotation onsets.
Adding metadata with 7 columns
120 matching events found
No baseline correction applied
0 projection items activated


<Epochs | 120 events (good & bad), -0.2 – 0.5 s (baseline off), ~141 KiB, data not loaded, with metadata,
 np.str_('1'): 20
 np.str_('2'): 20
 np.str_('3'): 15
 np.str_('4'): 15
 np.str_('5'): 15
 np.str_('6'): 20
 np.str_('7'): 15>

In [57]:
picks = [
    ii
    for ii, ch_name in enumerate(montage.ch_names)
    if any([elec in ch_name for elec in filtered_electrodes])
]
""" labels = (
    "ctx-rh-parahippocampal",
    "ctx-rh-fusiform",
    'Right-Inf-Lat-Vent', 
    'Right-Hippocampus'
) """
trans = mne.transforms.Transform('mri', 'head') 
fig = mne.viz.plot_alignment(
    mne.pick_info(epochs.info, picks),
    trans=trans,         
    subject=fs_subject,
    subjects_dir=subjects_dir,
    surfaces=[],
    coord_frame="mri",
)

brain = mne.viz.Brain(
    fs_subject,
    alpha=0.1,
    cortex="low_contrast",
    subjects_dir=str(fs_subjects_dir), 
    units="m",
    figure=fig,
)
brain.add_volume_labels(aseg="aparc+aseg", labels=regions_of_interest)

brain.show_view(azimuth=120, elevation=90, distance=0.5, )
brain.save_image("figures/regions_mydata01.png")

brain.show_view(azimuth=60, elevation=90, distance=0.5)
brain.save_image("figures/regions_mydata02.png")

brain.show_view(azimuth=60, elevation=45, distance=0.5)
brain.save_image("figures/regions_mydata03.png")

brain.show_view(azimuth=0, elevation=45, distance=0.5)
brain.save_image("figures/regions_mydata04.png")

brain.show_view(azimuth=30, elevation=45, distance=0.5)
brain.save_image("figures/regions_mydata05.png")

Channel types::	seeg: 22


Could not estimate rigid Talairach alignment, using identity matrix


A view with name (P_0x7f529c5a2350_9) is already registered
 => returning previous one


    Smoothing by a factor of 0.9
